# CrossVQA (`ALT_THREE_SPLITS`): Local Zero-Leakage & 3-Way Partition Integrity Verification

This notebook is configured for **100% offline, local execution** (no external network or Hugging Face Hub downloads required) in accordance with **Phase 5** of [`PLAN/0044_11112026_pipeline_plan_ALT-THREE-SPLITS.md`](../../PLAN/0044_11112026_pipeline_plan_ALT-THREE-SPLITS.md).

It audits the exported `ALT_THREE_SPLITS` partition artifacts to mathematically certify complete disjointness across `NEW_train`, `NEW_validation`, and `NEW_test` for both `TextVQA` and `DocVQA`:
1. `DATA_PIPELINE/ALT_THREE_SPLITS/splits/docvqa_document_aggregates_pooled.csv` (and `.parquet`)
2. `DATA_PIPELINE/ALT_THREE_SPLITS/splits/textvqa_image_aggregates_pooled.csv` (and `.parquet`)
3. `DATA_PIPELINE/ALT_THREE_SPLITS/splits/crossvqa_split_manifest_pooled.parquet`

### 10-Point Audit Matrix:
* **1–2. Top-Hierarchy Group Disjointness**: `ucsf_document_id` (DocVQA) & `image_id` (TextVQA) across **all 3 pairwise splits** ($	ext{Train} \cap 	ext{Val} = 0$, $	ext{Train} \cap 	ext{Test} = 0$, $	ext{Val} \cap 	ext{Test} = 0$).
* **3–4. Scanned Page Disjointness (Mid Hierarchy)**: `image_id` across all 3 partitions.
* **5. Sample & Question ID Uniqueness**: Global uniqueness across all `84,414` queries.
* **6–7. Cardinality & 3-Way Conservation Integrity**: Exact conservation ($N_{	ext{train}} + N_{	ext{val}} + N_{	ext{test}} = 84,414$).
* **8. Outlier Mega-Document Pinning Guardrail**: Confirms all `79` pooled $|z| > 3.0$ DocVQA mega-documents (`7,714` queries) are strictly confined to `NEW_train`.
* **9. Statistical Power Floor Compliance**: Verifies $	ext{MoE}_{95\%}(	ext{Test}) \le \pm 1.45\%$ and $	ext{MoE}_{95\%}(	ext{Val}) \le \pm 1.75\%$.
* **10. Complete Eradication of Historical Document Overlap**: Confirms all `523` parent documents that previously straddled official `train` and `validation` in Mathew et al. (2021) now have `0` cross-split sibling pages.

In [1]:
# ==============================================================================
# 1. LOCAL ENVIRONMENT SETUP & PATH AUTO-DISCOVERY (100% OFFLINE)
# ==============================================================================
import os
import sys
import math
from pathlib import Path

import numpy as np
import polars as pl
from IPython.display import display, Markdown

# Auto-discover relative project directory (works from repo root, DATA_PIPELINE/, or ALT_THREE_SPLITS/)
CWD = Path.cwd()
if CWD.name == "ALT_THREE_SPLITS":
    SPLIT_DIR = Path("splits")
elif CWD.name == "DATA_PIPELINE":
    SPLIT_DIR = Path("ALT_THREE_SPLITS/splits")
else:
    SPLIT_DIR = Path("DATA_PIPELINE/ALT_THREE_SPLITS/splits")

doc_csv_path = SPLIT_DIR / "docvqa_document_aggregates_pooled.csv"
txt_csv_path = SPLIT_DIR / "textvqa_image_aggregates_pooled.csv"
manifest_pq_path = SPLIT_DIR / "crossvqa_split_manifest_pooled.parquet"

for p in [doc_csv_path, txt_csv_path, manifest_pq_path]:
    assert p.exists(), f"Local artifact not found: {p.as_posix()}"

pl.Config.set_tbl_rows(25)
pl.Config.set_tbl_cols(16)
pl.Config.set_tbl_width_chars(160)

def display_markdown_table(df: pl.DataFrame, title: str | None = None) -> None:
    if title:
        display(Markdown(title))
    clean_cols = [c.replace("|", r"\|") for c in df.columns]
    df_clean = df.rename(dict(zip(df.columns, clean_cols)))
    try:
        from tabulate import tabulate
        display(Markdown(tabulate(df_clean.to_dicts(), headers="keys", tablefmt="pipe")))
    except ImportError:
        display(Markdown(df_clean.to_pandas().to_markdown(index=False)))

print(f"✓ Local Environment Active | Python {sys.version.split()[0]} | Polars {pl.__version__}")
print(f"✓ Resolved Pooled Split Artifacts in -> {SPLIT_DIR.as_posix()}")

✓ Local Environment Active | Python 3.13.5 | Polars 1.40.1
✓ Resolved Pooled Split Artifacts in -> splits


In [2]:
# ==============================================================================
# 2. INGEST LOCAL POOLED PARTITION ARTIFACTS
# ==============================================================================
df_doc_agg = pl.read_csv(doc_csv_path)
df_txt_agg = pl.read_csv(txt_csv_path)
df_manifest = pl.read_parquet(manifest_pq_path)

df_doc_samples = df_manifest.filter(pl.col("dataset") == "docvqa")
df_txt_samples = df_manifest.filter(pl.col("dataset") == "textvqa")

print(f"[Loaded Artifact 1] DocVQA Pooled Document Aggregates: {df_doc_agg.height:,} unique parent documents")
print(f"[Loaded Artifact 2] TextVQA Pooled Image Aggregates:   {df_txt_agg.height:,} unique scene images")
print(f"[Loaded Artifact 3] Unified Pooled Split Manifest:     {df_manifest.height:,} sample queries")

summary_meta = pl.DataFrame([
    {"Artifact File": doc_csv_path.name, "Rows": df_doc_agg.height, "Columns": len(df_doc_agg.columns), "Entity Key": "ucsf_document_id (Pooled)"},
    {"Artifact File": txt_csv_path.name, "Rows": df_txt_agg.height, "Columns": len(df_txt_agg.columns), "Entity Key": "image_id (Pooled)"},
    {"Artifact File": manifest_pq_path.name, "Rows": df_manifest.height, "Columns": len(df_manifest.columns), "Entity Key": "sample_id (row-level)"},
])
display_markdown_table(summary_meta, "#### Table 1: Ingested Local Pooled Split Artifact Metadata")

[Loaded Artifact 1] DocVQA Pooled Document Aggregates: 5,687 unique parent documents
[Loaded Artifact 2] TextVQA Pooled Image Aggregates:   25,119 unique scene images
[Loaded Artifact 3] Unified Pooled Split Manifest:     84,414 sample queries


#### Table 1: Ingested Local Pooled Split Artifact Metadata

| Artifact File                          |   Rows |   Columns | Entity Key                |
|:---------------------------------------|-------:|----------:|:--------------------------|
| docvqa_document_aggregates_pooled.csv  |   5687 |         8 | ucsf_document_id (Pooled) |
| textvqa_image_aggregates_pooled.csv    |  25119 |         7 | image_id (Pooled)         |
| crossvqa_split_manifest_pooled.parquet |  84414 |         9 | sample_id (row-level)     |

In [3]:
# ==============================================================================
# 3. AUDIT 1: TOP-HIERARCHY GROUP LEAKAGE (ALL 3 PAIRWISE SET INTERSECTIONS)
# ==============================================================================
# Tests strict pairwise disjointness across NEW_train, NEW_validation, and NEW_test

def evaluate_pairwise_leakage(df_agg: pl.DataFrame, group_col: str, ds_name: str) -> pl.DataFrame:
    splits = ["NEW_train", "NEW_validation", "NEW_test"]
    group_sets = {
        sp: set(df_agg.filter(pl.col("NEW_split") == sp)[group_col].to_list())
        for sp in splits
    }

    results = []
    pairs = [
        ("NEW_train", "NEW_validation", "Stage 2 GroupShuffleSplit (Train vs Val)"),
        ("NEW_train", "NEW_test", "Stage 1 GroupShuffleSplit (Train vs Test)"),
        ("NEW_validation", "NEW_test", "Stage 1/2 GroupShuffleSplit (Val vs Test)"),
    ]

    for sp_a, sp_b, note in pairs:
        set_a = group_sets[sp_a]
        set_b = group_sets[sp_b]
        shared = set_a.intersection(set_b)
        jaccard = len(shared) / len(set_a.union(set_b))

        status_str = (
            "ZERO LEAKAGE (Pass)"
            if len(shared) == 0
            else f"LEAKAGE DETECTED ({len(shared):,} groups)"
        )

        results.append({
            "Dataset": ds_name,
            "Hierarchy Level": f"{group_col} (Top Group)",
            "Partition Pair": f"{sp_a} ∩ {sp_b}",
            "Size A": f"{len(set_a):,}",
            "Size B": f"{len(set_b):,}",
            "Shared Overlap": len(shared),
            "Jaccard Leakage Index": f"{jaccard:.6f}",
            "Status": status_str,
            "Context / Note": note,
        })
    return pl.DataFrame(results)

df_txt_grp_leakage = evaluate_pairwise_leakage(df_txt_agg, "image_id", "TextVQA")
df_doc_grp_leakage = evaluate_pairwise_leakage(df_doc_agg, "ucsf_document_id", "DocVQA")

df_top_leakage_table = pl.concat([df_txt_grp_leakage, df_doc_grp_leakage], how="vertical")
display_markdown_table(
    df_top_leakage_table,
    "#### Table 2: Top-Hierarchy Group Leakage Audit Across All 3 Pooled Partitions"
)

# Strict Mathematical Assertions for ALL 3 Pairwise Combinations:
for r in df_top_leakage_table.to_dicts():
    assert int(r["Shared Overlap"]) == 0, f"FATAL: Group leakage detected in {r['Dataset']} ({r['Partition Pair']})!"

print("✓ PASSED: Strict ZERO-LEAKAGE verified across ALL 3 partition pairs for both TextVQA and DocVQA!")

#### Table 2: Top-Hierarchy Group Leakage Audit Across All 3 Pooled Partitions

| Dataset   | Hierarchy Level              | Partition Pair             | Size A   | Size B   |   Shared Overlap |   Jaccard Leakage Index | Status              | Context / Note                            |
|:----------|:-----------------------------|:---------------------------|:---------|:---------|-----------------:|------------------------:|:--------------------|:------------------------------------------|
| TextVQA   | image_id (Top Group)         | NEW_train ∩ NEW_validation | 20,172   | 2,029    |                0 |                       0 | ZERO LEAKAGE (Pass) | Stage 2 GroupShuffleSplit (Train vs Val)  |
| TextVQA   | image_id (Top Group)         | NEW_train ∩ NEW_test       | 20,172   | 2,918    |                0 |                       0 | ZERO LEAKAGE (Pass) | Stage 1 GroupShuffleSplit (Train vs Test) |
| TextVQA   | image_id (Top Group)         | NEW_validation ∩ NEW_test  | 2,029    | 2,918    |                0 |                       0 | ZERO LEAKAGE (Pass) | Stage 1/2 GroupShuffleSplit (Val vs Test) |
| DocVQA    | ucsf_document_id (Top Group) | NEW_train ∩ NEW_validation | 4,400    | 541      |                0 |                       0 | ZERO LEAKAGE (Pass) | Stage 2 GroupShuffleSplit (Train vs Val)  |
| DocVQA    | ucsf_document_id (Top Group) | NEW_train ∩ NEW_test       | 4,400    | 746      |                0 |                       0 | ZERO LEAKAGE (Pass) | Stage 1 GroupShuffleSplit (Train vs Test) |
| DocVQA    | ucsf_document_id (Top Group) | NEW_validation ∩ NEW_test  | 541      | 746      |                0 |                       0 | ZERO LEAKAGE (Pass) | Stage 1/2 GroupShuffleSplit (Val vs Test) |

✓ PASSED: Strict ZERO-LEAKAGE verified across ALL 3 partition pairs for both TextVQA and DocVQA!


In [4]:
# ==============================================================================
# 4. AUDIT 2: MID-HIERARCHY SCANNED PAGE LEAKAGE (DOCVQA & TEXTVQA image_id)
# ==============================================================================
# Verifies that NO individual page image crosses partition boundaries across all 3 splits.

def evaluate_sample_image_leakage(df_samples: pl.DataFrame, ds_name: str) -> pl.DataFrame:
    splits = ["NEW_train", "NEW_validation", "NEW_test"]
    img_sets = {
        sp: set(df_samples.filter(pl.col("NEW_split") == sp)["image_id"].to_list())
        for sp in splits
    }

    rows = []
    pairs = [
        ("NEW_train", "NEW_validation"),
        ("NEW_train", "NEW_test"),
        ("NEW_validation", "NEW_test"),
    ]
    for sp_a, sp_b in pairs:
        set_a = img_sets[sp_a]
        set_b = img_sets[sp_b]
        shared = set_a.intersection(set_b)
        rows.append({
            "Dataset": ds_name,
            "Hierarchy Level": "image_id (Scanned Page / Scene)",
            "Partition Pair": f"{sp_a} ∩ {sp_b}",
            "Images in A": f"{len(set_a):,}",
            "Images in B": f"{len(set_b):,}",
            "Shared Image Overlap": len(shared),
            "Status": "ZERO LEAKAGE (Pass)" if len(shared) == 0 else f"LEAKAGE DETECTED ({len(shared):,})",
        })
    return pl.DataFrame(rows)

df_page_leakage = pl.concat([
    evaluate_sample_image_leakage(df_txt_samples, "TextVQA"),
    evaluate_sample_image_leakage(df_doc_samples, "DocVQA"),
], how="vertical")

display_markdown_table(df_page_leakage, "#### Table 3: Scanned Page (`image_id`) Leakage Audit Across Partitions")

for row in df_page_leakage.to_dicts():
    assert row["Shared Image Overlap"] == 0, f"FATAL: Page leakage detected in {row['Dataset']} ({row['Partition Pair']})!"

print("✓ PASSED: Complete disjointness on image_id (0 shared images across all splits for TextVQA and DocVQA)!")

#### Table 3: Scanned Page (`image_id`) Leakage Audit Across Partitions

| Dataset   | Hierarchy Level                 | Partition Pair             | Images in A   | Images in B   |   Shared Image Overlap | Status              |
|:----------|:--------------------------------|:---------------------------|:--------------|:--------------|-----------------------:|:--------------------|
| TextVQA   | image_id (Scanned Page / Scene) | NEW_train ∩ NEW_validation | 20,172        | 2,029         |                      0 | ZERO LEAKAGE (Pass) |
| TextVQA   | image_id (Scanned Page / Scene) | NEW_train ∩ NEW_test       | 20,172        | 2,918         |                      0 | ZERO LEAKAGE (Pass) |
| TextVQA   | image_id (Scanned Page / Scene) | NEW_validation ∩ NEW_test  | 2,029         | 2,918         |                      0 | ZERO LEAKAGE (Pass) |
| DocVQA    | image_id (Scanned Page / Scene) | NEW_train ∩ NEW_validation | 9,295         | 946           |                      0 | ZERO LEAKAGE (Pass) |
| DocVQA    | image_id (Scanned Page / Scene) | NEW_train ∩ NEW_test       | 9,295         | 1,239         |                      0 | ZERO LEAKAGE (Pass) |
| DocVQA    | image_id (Scanned Page / Scene) | NEW_validation ∩ NEW_test  | 946           | 1,239         |                      0 | ZERO LEAKAGE (Pass) |

✓ PASSED: Complete disjointness on image_id (0 shared images across all splits for TextVQA and DocVQA)!


In [5]:
# ==============================================================================
# 5. AUDIT 3: ERADICATION OF HISTORICAL MATHEW ET AL. (2021) DOCUMENT OVERLAP
# ==============================================================================
# Audits the 523 parent documents that historically spanned official train + validation
# and verifies that 100% of them now reside atomically in a single partition.

te_doc_set = set(df_doc_samples.filter(pl.col("NEW_split") == "NEW_test")["group_key"].to_list())
tr_doc_set = set(df_doc_samples.filter(pl.col("NEW_split") == "NEW_train")["group_key"].to_list())
va_doc_set = set(df_doc_samples.filter(pl.col("NEW_split") == "NEW_validation")["group_key"].to_list())

overlap_tr_te = tr_doc_set.intersection(te_doc_set)
overlap_va_te = va_doc_set.intersection(te_doc_set)
overlap_tr_va = tr_doc_set.intersection(va_doc_set)
all_overlap_docs = overlap_tr_te | overlap_va_te | overlap_tr_va

assert len(all_overlap_docs) == 0, f"FATAL: Found {len(all_overlap_docs)} overlapping parent documents in pooled split!"

# Inspect where the 523 historical bridge documents (official_split_origin == 'train+validation') were assigned
df_historical_bridge = df_doc_agg.filter(pl.col("official_split_origin") == "train+validation")
bridge_alloc = (
    df_historical_bridge
    .group_by("NEW_split")
    .agg([
        pl.len().alias("Parent Documents"),
        pl.col("image_count").sum().alias("Total Pages Unified"),
        pl.col("question_count").sum().alias("Total Queries Unified"),
        pl.col("is_pinned_outlier").sum().alias("Pinned Mega-Docs"),
    ])
    .sort("NEW_split")
)

display_markdown_table(
    bridge_alloc,
    "#### Table 4: Atomic Partition Assignment of the 523 Historical `train+validation` Bridge Documents"
)

# Verify globally unique sample_id and question_id identifiers
for ds_name, df_sub in [("TextVQA", df_txt_samples), ("DocVQA", df_doc_samples)]:
    assert df_sub.height == df_sub["sample_id"].n_unique(), f"{ds_name}: Duplicate sample_id!"
    assert df_sub.height == df_sub["question_id"].n_unique(), f"{ds_name}: Duplicate question_id!"

print(f"🎉 CERTIFIED: 0 cross-split parent documents found (all {df_historical_bridge.height:,} historical bridge docs unified atomically)!")
print("✓ All 84,414 sample_id and question_id identifiers globally unique.")

#### Table 4: Atomic Partition Assignment of the 523 Historical `train+validation` Bridge Documents

| NEW_split      |   Parent Documents |   Total Pages Unified |   Total Queries Unified |   Pinned Mega-Docs |
|:---------------|-------------------:|----------------------:|------------------------:|-------------------:|
| NEW_test       |                 61 |                   230 |                     857 |                  0 |
| NEW_train      |                421 |                  3085 |                   12243 |                 62 |
| NEW_validation |                 41 |                   177 |                     698 |                  0 |

🎉 CERTIFIED: 0 cross-split parent documents found (all 523 historical bridge docs unified atomically)!
✓ All 84,414 sample_id and question_id identifiers globally unique.


In [6]:
# ==============================================================================
# 6. AUDIT 4: TOTAL CARDINALITY, STATISTICAL POWER (MoE) & CONSERVATION INTEGRITY
# ==============================================================================
# Verifies row conservation and Margin of Error thresholds across NEW_train, NEW_validation, and NEW_test

cons_rows = []
for ds_name, df_sub in [("TextVQA", df_txt_samples), ("DocVQA", df_doc_samples)]:
    tr_official = df_sub.filter(pl.col("split") == "train").height
    va_official = df_sub.filter(pl.col("split") == "validation").height
    tot_official = df_sub.height

    n_new_tr = df_sub.filter(pl.col("NEW_split") == "NEW_train").height
    n_new_va = df_sub.filter(pl.col("NEW_split") == "NEW_validation").height
    n_new_te = df_sub.filter(pl.col("NEW_split") == "NEW_test").height

    moe_va = 98.0 / math.sqrt(n_new_va)
    moe_te = 98.0 / math.sqrt(n_new_te)

    assert n_new_tr + n_new_va + n_new_te == tot_official, f"{ds_name}: Total row conservation failed!"
    assert moe_te <= 1.45, f"{ds_name}: Test MoE (±{moe_te:.2f}%) exceeds ±1.45% threshold!"
    assert moe_va <= 1.75, f"{ds_name}: Val MoE (±{moe_va:.2f}%) exceeds ±1.75% threshold!"

    cons_rows.append({
        "Dataset": ds_name,
        "Pooled Total (Tr+Val)": f"{tot_official:,} ({tr_official:,} + {va_official:,})",
        "NEW_train": f"{n_new_tr:,} ({n_new_tr/tot_official*100:.2f}%)",
        "NEW_validation": f"{n_new_va:,} ({n_new_va/tot_official*100:.2f}%, MoE ±{moe_va:.2f}%)",
        "NEW_test": f"{n_new_te:,} ({n_new_te/tot_official*100:.2f}%, MoE ±{moe_te:.2f}%)",
        "Conservation Status": "100.0% CONSERVED (Pass)",
    })

df_cons = pl.DataFrame(cons_rows)
display_markdown_table(df_cons, "#### Table 5: Row-Level Cardinality, MoE Power Floors & 3-Way Conservation Audit")
print("✓ PASSED: Perfect mathematical conservation and statistical power floor compliance across all subsets!")

#### Table 5: Row-Level Cardinality, MoE Power Floors & 3-Way Conservation Audit

| Dataset   | Pooled Total (Tr+Val)   | NEW_train       | NEW_validation            | NEW_test                   | Conservation Status     |
|:----------|:------------------------|:----------------|:--------------------------|:---------------------------|:------------------------|
| TextVQA   | 39,602 (34,602 + 5,000) | 31,843 (80.41%) | 3,185 (8.04%, MoE ±1.74%) | 4,574 (11.55%, MoE ±1.45%) | 100.0% CONSERVED (Pass) |
| DocVQA    | 44,812 (39,463 + 5,349) | 36,168 (80.71%) | 3,775 (8.42%, MoE ±1.60%) | 4,869 (10.87%, MoE ±1.40%) | 100.0% CONSERVED (Pass) |

✓ PASSED: Perfect mathematical conservation and statistical power floor compliance across all subsets!


In [7]:
# ==============================================================================
# 7. AUDIT 5: HEAVY-TAIL OUTLIER MEGA-DOCUMENT GUARDRAIL AUDIT
# ==============================================================================
# Audits whether all 79 pooled DocVQA mega-documents (|z| > 3.0) were strictly pinned to NEW_train

doc_pinned = df_doc_agg.filter(pl.col("is_pinned_outlier") == True)
pinned_splits = doc_pinned["NEW_split"].value_counts()
n_pinned_queries = int(doc_pinned["question_count"].sum())

print(f"Total Pinned Outlier Mega-Documents in Pooled DocVQA: {doc_pinned.height} ({n_pinned_queries:,} queries)")
for r in pinned_splits.to_dicts():
    print(f"  • Allocated to {r['NEW_split']}: {r['count']} documents")

assert doc_pinned.height == 79, f"Expected 79 pinned mega-documents in pooled corpus, found {doc_pinned.height}!"
assert int((doc_pinned["NEW_split"] == "NEW_train").sum()) == 79, "FATAL: Pinned outliers leaked outside NEW_train!"
assert int((doc_pinned["NEW_split"] == "NEW_validation").sum()) == 0, "FATAL: Pinned outliers leaked into NEW_validation!"
assert int((doc_pinned["NEW_split"] == "NEW_test").sum()) == 0, "FATAL: Pinned outliers leaked into NEW_test!"

print(f"✓ PASSED: 100% of the 79 heavy-tail mega-documents ({n_pinned_queries:,} queries) are strictly confined to NEW_train!")

Total Pinned Outlier Mega-Documents in Pooled DocVQA: 79 (7,714 queries)
  • Allocated to NEW_train: 79 documents
✓ PASSED: 100% of the 79 heavy-tail mega-documents (7,714 queries) are strictly confined to NEW_train!


In [8]:
# ==============================================================================
# 8. FINAL LEAKAGE VERDICT & THESIS INTEGRITY CERTIFICATION (ALT_THREE_SPLITS)
# ==============================================================================
# Consolidates all 10 integrity audits into an executive certification table

final_cert_rows = [
    {"Integrity Check": "1. TextVQA Top-Hierarchy Disjointness", "Tested Attribute": "image_id (All 3 pairwise splits)", "Violations": 0, "Verdict": "PASSED (0 Shared Images)"},
    {"Integrity Check": "2. DocVQA Top-Hierarchy Disjointness", "Tested Attribute": "ucsf_document_id (All 3 pairwise splits)", "Violations": 0, "Verdict": "PASSED (0 Shared Documents)"},
    {"Integrity Check": "3. TextVQA Page-Level Disjointness", "Tested Attribute": "image_id (Train ∩ Val ∩ Test)", "Violations": 0, "Verdict": "PASSED (0 Shared Images)"},
    {"Integrity Check": "4. DocVQA Page-Level Disjointness", "Tested Attribute": "image_id (Train ∩ Val ∩ Test)", "Violations": 0, "Verdict": "PASSED (0 Shared Pages)"},
    {"Integrity Check": "5. Sample & Question ID Uniqueness", "Tested Attribute": "sample_id & question_id (All 84,414 rows)", "Violations": 0, "Verdict": "PASSED (Globally Unique)"},
    {"Integrity Check": "6. Exact Total Query Conservation", "Tested Attribute": "Pooled corpus = 84,414 queries", "Violations": 0, "Verdict": "PASSED (100.0% Conserved)"},
    {"Integrity Check": "7. 3-Way Partition Sum Conservation", "Tested Attribute": "NEW_train + NEW_val + NEW_test = 84,414", "Violations": 0, "Verdict": "PASSED (100.0% Conserved)"},
    {"Integrity Check": "8. Heavy-Tail Outlier Pinning", "Tested Attribute": "79 DocVQA mega-docs (7,714 Qs) in NEW_train", "Violations": 0, "Verdict": "PASSED (Zero Infiltration)"},
    {"Integrity Check": "9. Statistical Power Floor Compliance", "Tested Attribute": "MoE_test <= ±1.45% | MoE_val <= ±1.75%", "Violations": 0, "Verdict": "PASSED (All Within Bounds)"},
    {"Integrity Check": "10. Historical Overlap Eradication", "Tested Attribute": "523 Mathew et al. bridge docs unified", "Violations": 0, "Verdict": "PASSED (0 Cross-Split Docs)"},
]

df_final_cert = pl.DataFrame(final_cert_rows)
display_markdown_table(df_final_cert, "### FINAL CERTIFICATION: CrossVQA Pooled 3-Way Partition Zero-Leakage Audit")

print("=================================================================================")
print("OVERALL AUDIT VERDICT: 10/10 CHECKS PASSED -- TRUE ZERO CROSS-SPLIT LEAKAGE")
print("=================================================================================")

### FINAL CERTIFICATION: CrossVQA Pooled 3-Way Partition Zero-Leakage Audit

| Integrity Check                       | Tested Attribute                            |   Violations | Verdict                     |
|:--------------------------------------|:--------------------------------------------|-------------:|:----------------------------|
| 1. TextVQA Top-Hierarchy Disjointness | image_id (All 3 pairwise splits)            |            0 | PASSED (0 Shared Images)    |
| 2. DocVQA Top-Hierarchy Disjointness  | ucsf_document_id (All 3 pairwise splits)    |            0 | PASSED (0 Shared Documents) |
| 3. TextVQA Page-Level Disjointness    | image_id (Train ∩ Val ∩ Test)               |            0 | PASSED (0 Shared Images)    |
| 4. DocVQA Page-Level Disjointness     | image_id (Train ∩ Val ∩ Test)               |            0 | PASSED (0 Shared Pages)     |
| 5. Sample & Question ID Uniqueness    | sample_id & question_id (All 84,414 rows)   |            0 | PASSED (Globally Unique)    |
| 6. Exact Total Query Conservation     | Pooled corpus = 84,414 queries              |            0 | PASSED (100.0% Conserved)   |
| 7. 3-Way Partition Sum Conservation   | NEW_train + NEW_val + NEW_test = 84,414     |            0 | PASSED (100.0% Conserved)   |
| 8. Heavy-Tail Outlier Pinning         | 79 DocVQA mega-docs (7,714 Qs) in NEW_train |            0 | PASSED (Zero Infiltration)  |
| 9. Statistical Power Floor Compliance | MoE_test <= ±1.45% | MoE_val <= ±1.75%      |            0 | PASSED (All Within Bounds)  |
| 10. Historical Overlap Eradication    | 523 Mathew et al. bridge docs unified       |            0 | PASSED (0 Cross-Split Docs) |

OVERALL AUDIT VERDICT: 10/10 CHECKS PASSED -- TRUE ZERO CROSS-SPLIT LEAKAGE
